In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
# Initialize Spark Session (if not already running)
spark = SparkSession.builder.appName("PySparkJoins").getOrCreate()
# Create sample DataFrames
df3 = spark.createDataFrame([(1, "Rabi", 23),(2, "Chabi", 24),(3, "gabi", 25)],["id", "name", "age"])

df4 = spark.createDataFrame([(1, "HR"),(2, "IT"),(3, "FINANCE")],["id", "departmennt"])

joined_df = df3.join(df4, on=["id"],how="left")

display(joined_df)

In [0]:
joined_df.explain()

In [0]:
data1 = [
    (1, "A"),
    (2, "B"),
    (3, "C")
]
data2 = [
    (1, "X"),
    (2, "Y"),
    (4, "Z")
]

df1 = spark.createDataFrame(data1, ["ID", "NAME"])
df2 = spark.createDataFrame(data2, ["ID", "DESC"])

# Inner Join
df_inner = df1.join(df2, on="ID", how="inner")
display(df_inner)

# Left Join
df_left = df1.join(df2, on="ID", how="left")
display(df_left)
# ID	  NAME	DESC
# 1	      A	    X
# 2	      B	    Y
# 3	      C	    null

# Right Join
df_right = df1.join(df2, on="ID", how="right")
display(df_right)

# Full Outer Join
df_outer = df1.join(df2, on="ID", how="outer")
display(df_outer)

# Left Semi Join
df_leftsemi = df1.join(df2, on="ID", how="left_semi")
display(df_leftsemi)
# ID	NAME
# 1	A
# 2	B

# Left Anti Join
df_leftanti = df1.join(df2, on="ID", how="left_anti")
display(df_leftanti)
# ID	NAME
# 3	C

# A left semi join returns only the rows from the left DataFrame that have a matching key in the right DataFrame. It does not include any columns from the right DataFrame. Also not include any row which is not matching from right datagrame

# A left anti: (opposite of left join) join returns only the rows from the left DataFrame that do not have a matching key in the right DataFrame. Again, only columns from the left DataFrame are included.

In [0]:
# Repartition and coalesce are used to change the number of partitions in a Spark DataFrame:

# repartition(n): Increases or decreases the number of partitions by shuffling all data. Use when you need more partitions or a balanced distribution.
# coalesce(n): Decreases the number of partitions without a full shuffle, combining existing partitions. Use when reducing partitions, especially after filtering.

data = [(1, "A"),(2, "B"),(3, "C"),(4, "D"),(5, "E"),(6, "F")]
df = spark.createDataFrame(data,["ID", "NAME"])

# Repartition to 4 partitions (full shuffle)
df_repart = df.repartition(4)
display(df_repart)

# Coalesce to 2 partitions (no full shuffle)
df_coal = df.coalesce(2)
display(df_coal)

In [0]:
# Repartition and coalesce 
# You will not see a difference in the displayed data, but the number of partitions in the underlying DataFrame will change.

# To check the number of partitions, you can use:
print(df_repart.rdd.getNumPartitions())
print(df_coal.rdd.getNumPartitions())

In [0]:
# A broadcast join is an efficient join strategy in Spark where a small DataFrame is sent to all worker nodes, allowing the join to be performed locally without shuffling large datasets. This is useful when one DataFrame is much smaller than the other.
#broadcast join
from pyspark.sql.functions import broadcast
df_broadcast = df1.join(broadcast(df2), on="ID", how="inner")
display(df_broadcast)

In [0]:
df3 =  joined_df

In [0]:
from pyspark.sql.functions import *


In [0]:
display(df3)
df3.orderBy("age", ascending = False).show()

In [0]:
display(df3.filter(col("age")>24))


In [0]:
display(df3.select(count("id")))

In [0]:
display(df3.select("id","age"))

In [0]:
# In PySpark, the DataFrame.show() method is used to display the contents of a DataFrame in a tabular format in the notebook or console. It prints a specified number of rows (default is 20) and is useful for quickly inspecting your data
df3.show(5)

In [0]:
data = [(1,"A"),(2,"B")]
df = spark.createDataFrame(data,["ID","NAME"])
df.show()


In [0]:
# add a column in df here 2 row inserted
df = df.withColumn("SALARY", lit(1000))
df = df.withColumn("SALARY", lit(2000))

df.show()


In [0]:
# rename
df = df.withColumnRenamed("SALARY","SALARYY")

In [0]:
# Handle null value in pyspark
data = [
    (1, "A", None),
    (2, None, 2000),
    (3, "C", 3000),
    (None, "D", None)
]
df = spark.createDataFrame(data,["ID", "NAME", "SALARY"])
df2 = df.show()

# Replace nulls in SALARY with 0 and NAME with "Unknown"
df_filled = df.fillna({"SALARY": 0,"NAME": "Unknown"})
display(df_filled)

# Drop rows where any column is null & drop specified column name
df_dropped = df.dropna()
df_dropped2 = df.dropna(subset=["NAME"])
display(df_dropped)
display(df_dropped2)

In [0]:
# read a csv data
df = spark.read.option("header","true").csv("path")
df.show()
# OR
df = spark.read.format("CSV").option("header","true").load("path")
df.show()

# write a DAta
# ignore: Does nothing if data already exists at the path; the write is skipped.
df = spark.write.mode("overwrite").parquet("path")
                                                        

In [0]:
data = [
    (1, "A", "IT", 50000),
    (2, "B", "IT", 60000),
    (3, "C", "IT", 60000),
    (4, "D", "HR", 40000),
    (5, "E", "HR", 45000),
    (6, "F", "IT", 70000)
]

columns = ["emp_id", "name", "dept", "salary"]

df = spark.createDataFrame(data, columns)

In [0]:
#
from pyspark.sql.functions import sum
display(df)
from pyspark.sql.window import Window
from pyspark.sql.functions import *
WindowSpec = Window.partitionBy('dept').orderBy(desc('salary'))
# df2 = df.withColumn("dense_rnk",dense_rank().over(WindowSpec))
# df3 = df.groupBy('dept').agg(sum('salary')).alias('total_sal')
# df4 = df.withColumn("row_number",row_number().over(WindowSpec))
df5 = df.withColumn("rnk",rank().over(WindowSpec))
display(df5)


In [0]:
# perfect example of aggregate and sum
from pyspark.sql.functions import sum
display(df)
df2 = df.groupBy('dept').agg(sum('salary'))
display(df2)

In [0]:
#Row Number
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, dense_rank

windowSpec = Window.partitionBy("dept").orderBy("salary")

df.withColumn("row_num", row_number().over(windowSpec)).show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
windowSpec = Window.partitionBy("dept").orderBy(df["salary"].desc())
df.withColumn("row_num",row_number().over(windowSpec)).show()


In [0]:

#RANK
from pyspark.sql.functions import rank

df.withColumn("rank", rank().over(windowSpec)).show()

In [0]:
# DEnse Rank
from pyspark.sql.functions import dense_rank

df.withColumn("dense_rank", dense_rank().over(windowSpec)).show()

In [0]:
#scd1
dim_df = spark.createDataFrame([
    (1, "Ravi", "Delhi"),
    (2, "Amit", "Pune")
], ["cust_id", "name", "city"])


In [0]:
# Incoming Source Data
src_df = spark.createDataFrame([
    (1, "Ravi", "Mumbai")   # city changed
], ["cust_id", "name", "city"])


In [0]:
# SCD TYPE 1 Logic (Overwrite) Old city Delhi is replaced by Mumbai
from pyspark.sql.functions import col
scd1_df = dim_df.alias("d") \
    .join(src_df.alias("s"), "cust_id", "left") \
    .select(
        "cust_id",
        col("s.name").alias("name"),
        col("s.city").alias("city")
    )


In [0]:
scd1_df.show()

In [0]:
joined_df = dim_df.alias("d") \
    .join(src_df.alias("s"), "cust_id", "left")
display(joined_df)    

In [0]:
#scd2 Existing Dimension Table
from pyspark.sql.functions import current_date, lit

dim_dff = spark.createDataFrame([
    (1, "Ravi", "Delhi", "2023-01-01", None, 1),
    (2, "Amit", "Pune", "2023-01-01", None, 1)
], ["cust_id", "name", "city", "start_date", "end_date", "is_current"])


In [0]:
# Incoming Source Data
src_df = spark.createDataFrame([
    (1, "Ravi", "Mumbai")  # city changed
], ["cust_id", "name", "city"])


In [0]:
expired_df = changed_df.withColumn(
    "end_date", current_date()
).withColumn(
    "is_current", lit(0)
)


In [0]:
#ex of zip
l1 = (1, 2, 3)
l2 = (4, 5, 6)

result = list(zip(l1, l2))
print(result)

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import when, col
# salary>10000 <20000 5%
# salary>20000 <30000 10%
# salary>30000 15%
# else 2%
# Sample DataFrame
data = [
    Row(salary=9000),
    Row(salary=15000),
    Row(salary=25000),
    Row(salary=35000)
]
df = spark.createDataFrame(data)
display(df)
# df_new = df.withColumn("bonus", 
# when (col("salary")>10000 and col("salary")<20000, (col("salary")*.05+col("salary")))\
# .when (col("salary")>20000 and col("salary")<30000, )
# )
df_new = df.withColumn(
    "bonus",
    when(
        (col("salary") > 10000) & (col("salary") < 20000),
        col("salary") * 0.05
    ).when(
        (col("salary") > 20000) & (col("salary") < 30000),
        col("salary") * 0.10
    ).when(
        col("salary") > 30000,
        col("salary") * 0.15
    ).otherwise(
        col("salary") * 0.02
    )
)
display(df_new)

In [0]:
salary	bonus
9000	180
15000	750
25000	2500
35000	5250

In [0]:
#DElOITTE How to convert array column into list of columns
from pyspark.sql.functions import col
data = [
    (1, ["Math", "Eng"]),
    (2, ["Sci", "Bio"])
]
df = spark.createDataFrame(data, ["id", "subjects"])
# df2 = df.withColumn("subject1", col("subjects")[0]\
#     .withColumn("subject2", col("subjects")[1]))
df2 = df.withColumn("subject1",col("subjects")[0]).withColumn("subject2",col("subjects")[1])    
display(df2['id',"subject1","subject2"])    
# df2 = df.select("id", explode("subjects").alias("subject")).show()

In [0]:
from pyspark.sql.functions import *
data = [
    ("John", "Smith"),
    ("Jane", "D!oe"),
    ("Bob", "Johnson"),
    ("Mike", "Pant$on"),
    ("Sarah", "Jones"),
    ("Tom", "L#ee")
]

df = spark.createDataFrame(data,["firstName", "lastName"])
# df2 = df.withColumn("lastnames_modified",regexp_replace(col("lastName"),r"[@!$#]",""))
df2 = df.withColumn("lastnames", when(col("lastName").contains("#"),"").otherwise(col("lastName")))
display(df)
display(df2)

In [0]:
from pyspark.sql.functions import col, when
emp_df.withColumn("adult", when(col("age")<18,"no").when(col("age")>18,"yes").otherwise("unknown")).show()